# Day 2 | Notebook 1: Async Ingestion & Scale
**RedisVL Focus:** `AsyncSearchIndex`, `asyncio.gather`, throughput benchmarking

---
## 📌 Learning Objectives
1. Understand why async matters in production RAG pipelines
2. Measure the real speedup of parallel ingestion
3. Build an async ingestion pipeline for 500+ documents
4. Benchmark throughput (docs/sec) as a professional metric

---

In [1]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False   # Set True during class demo to reveal deeper insights
REDIS_URL = "redis://redis-vector-db:6379"   # Docker service name
# ─────────────────────────────────────────────────────────────────

In [2]:
# Cell 1: Install & imports
import asyncio
import time
import numpy as np
from redisvl.index import AsyncSearchIndex, SearchIndex
from redisvl.schema import IndexSchema

print("✅ Imports OK")

✅ Imports OK


## 📌 Concept: Blocking vs Non-Blocking I/O

When your program writes data to Redis:
- **Sync** (blocking): send → wait for reply → send next → wait → ...
- **Async** (non-blocking): send, send, send → collect all replies at once

For 1,000 documents with 5ms network latency each:
```
Sync:  1000 × 5ms = 5,000ms = 5 seconds
Async: max(all 5ms waits in parallel) ≈ 5-50ms
```

Python's `asyncio` is the engine. RedisVL's `AsyncSearchIndex` is the interface.

In [3]:
# Cell 2: Define our index schema
SCHEMA_DICT = {
    "index": {
        "name": "nb01_products",
        "prefix": "product:",
        "storage_type": "json"
    },
    "fields": [
        {"name": "title",       "type": "text"},
        {"name": "category",    "type": "tag"},
        {"name": "price",       "type": "numeric"},
        {"name": "embedding",   "type": "vector",
         "attrs": {"dims": 32, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
schema = IndexSchema.from_dict(SCHEMA_DICT)
print("✅ Schema defined with fields:", [f.name for f in schema.fields.values()])

✅ Schema defined with fields: ['title', 'category', 'price', 'embedding']


In [4]:
# Cell 3: Generate synthetic product data
# (Using small random vectors — NB04 uses real HFTextVectorizer embeddings)

def generate_products(n: int) -> list[dict]:
    categories = ["laptop", "tablet", "audio", "accessories"]
    products = []
    for i in range(n):
        products.append({
            "title":     f"Product-{i:04d}",
            "category":  categories[i % len(categories)],
            "price":     round(100 + (i % 50) * 20.0, 2),
            "embedding": np.random.rand(32).astype(np.float32).tolist()
        })
    return products

products = generate_products(500)
print(f"✅ Generated {len(products)} products")
print(f"   Sample: {products[0]}")

✅ Generated 500 products
   Sample: {'title': 'Product-0000', 'category': 'laptop', 'price': 100.0, 'embedding': [0.2888861894607544, 0.06444215774536133, 0.8487359285354614, 0.1139085590839386, 0.08760014921426773, 0.39198318123817444, 0.5885430574417114, 0.7999693751335144, 0.009637963958084583, 0.986777126789093, 0.4263211786746979, 0.0036836734507232904, 0.5035457015037537, 0.10360537469387054, 0.13502460718154907, 0.14157764613628387, 0.7937300205230713, 0.2138122022151947, 0.8414541482925415, 0.3062607944011688, 0.23155993223190308, 0.20303118228912354, 0.41633325815200806, 0.7232411503791809, 0.12085126340389252, 0.6779785752296448, 0.584860622882843, 0.5418819189071655, 0.2665194272994995, 0.7058317065238953, 0.7352850437164307, 0.8513978719711304]}


## 🔍 Demo: Synchronous Ingestion (Baseline)

In [5]:
# Cell 4: Sync ingestion — measure time
sync_index = SearchIndex(schema, redis_url=REDIS_URL)
sync_index.create(overwrite=True)

t0 = time.perf_counter()
# Load in 10 batches of 50 — synchronously one by one
BATCH_SIZE = 50
for i in range(0, len(products), BATCH_SIZE):
    batch = products[i:i + BATCH_SIZE]
    sync_index.load(batch)
    
sync_time = time.perf_counter() - t0
sync_throughput = len(products) / sync_time

print(f"🐢 SYNC Ingestion:")
print(f"   Documents : {len(products)}")
print(f"   Time      : {sync_time:.3f}s")
print(f"   Throughput: {sync_throughput:.0f} docs/sec")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Each sync batch waits for Redis to ACK before sending the next.")
    print("   This is safe but wasteful — the network is idle between writes.")

🐢 SYNC Ingestion:
   Documents : 500
   Time      : 0.040s
   Throughput: 12595 docs/sec


## 🔍 Demo: Asynchronous Ingestion with `AsyncSearchIndex`

In [6]:
# Cell 5: Define the async ingestion coroutine
async def load_batch_async(index: AsyncSearchIndex, batch: list) -> int:
    """Load one batch asynchronously and return the batch size."""
    await index.load(batch)
    return len(batch)

print("✅ Coroutine defined: load_batch_async")
print("   'async def' tells Python this function can be paused")
print("   'await index.load()' pauses HERE while Redis writes")

✅ Coroutine defined: load_batch_async
   'async def' tells Python this function can be paused
   'await index.load()' pauses HERE while Redis writes


In [7]:
# Cell 6: Run all batches in parallel — measure time
async def run_async_ingestion(products: list, batch_size: int = 50):
    async_index = AsyncSearchIndex(schema, redis_url=REDIS_URL)
    await async_index.create(overwrite=True)

    # Split into batches
    batches = [products[i:i+batch_size] for i in range(0, len(products), batch_size)]
    print(f"   Batches to process: {len(batches)} × {batch_size} docs")

    t0 = time.perf_counter()
    # asyncio.gather = run ALL coroutines CONCURRENTLY
    results = await asyncio.gather(*[load_batch_async(async_index, b) for b in batches])
    elapsed = time.perf_counter() - t0

    await async_index.client.aclose()
    return sum(results), elapsed

total_docs, async_time = await run_async_ingestion(products)
async_throughput = total_docs / async_time
speedup = sync_time / async_time

print(f"\n⚡ ASYNC Ingestion:")
print(f"   Documents : {total_docs}")
print(f"   Time      : {async_time:.3f}s")
print(f"   Throughput: {async_throughput:.0f} docs/sec")
print(f"\n📊 Speedup: {speedup:.1f}× faster than sync")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: asyncio.gather() submits all 10 batch coroutines at once.")
    print("   Python's event loop runs whichever batch gets a Redis ACK first.")
    print("   No thread is created — this is cooperative multitasking.")

   Batches to process: 10 × 50 docs

⚡ ASYNC Ingestion:
   Documents : 500
   Time      : 0.059s
   Throughput: 8495 docs/sec

📊 Speedup: 0.7× faster than sync


In [8]:
# Cell 7: Async VectorQuery — search while also being non-blocking
from redisvl.query import VectorQuery

async def async_search_demo():
    async_index = AsyncSearchIndex(schema, redis_url=REDIS_URL)
    # Note: we re-use the index created earlier (no overwrite)

    query_vec = np.random.rand(32).astype(np.float32).tolist()
    vq = VectorQuery(
        vector=query_vec,
        vector_field_name="embedding",
        return_fields=["title", "category", "price"],
        num_results=3
    )

    results = await async_index.query(vq)
    await async_index.client.aclose()
    return results

results = await async_search_demo()
print("🔍 Top 3 results (async query):")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['title']} | {r['category']} | ${r['price']} | dist={float(r['vector_distance']):.4f}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: 'await async_index.query()' is the same as sync query but")
    print("   it yields control while Redis processes the ANN search.")
    print("   Crucial in FastAPI where many users query simultaneously.")

🔍 Top 3 results (async query):
  1. Product-0353 | tablet | $160 | dist=0.1121
  2. Product-0353 | tablet | $160 | dist=0.1121
  3. Product-0034 | audio | $780 | dist=0.1185


In [9]:
# Cell 8: Benchmark — parallel queries (QPS = Queries Per Second)
async def parallel_query_benchmark(n_queries: int = 50):
    async_index = AsyncSearchIndex(schema, redis_url=REDIS_URL)

    async def single_query(_):
        vec = np.random.rand(32).astype(np.float32).tolist()
        vq = VectorQuery(vec, "embedding", return_fields=["title"], num_results=1)
        return await async_index.query(vq)

    t0 = time.perf_counter()
    await asyncio.gather(*[single_query(i) for i in range(n_queries)])
    elapsed = time.perf_counter() - t0

    await async_index.client.aclose()
    return n_queries / elapsed

qps = await parallel_query_benchmark(50)
print(f"⚡ Query Throughput: {qps:.0f} QPS (queries per second)")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: QPS is the standard metric for search system benchmarks.")
    print("   Industry target: >100 QPS for a production API with <100ms latency.")
    print("   Redis FLAT index: O(n) scan. HNSW index: O(log n) — much faster at scale.")

⚡ Query Throughput: 815 QPS (queries per second)


## 🛠️ Practice: Batch Size Tuning

In [10]:
# Cell 9: Try different batch sizes and compare throughput
# Run this cell as-is and observe the results table

async def benchmark_batch_size(batch_size: int) -> float:
    schema_b = IndexSchema.from_dict(SCHEMA_DICT)
    async_index = AsyncSearchIndex(schema_b, redis_url=REDIS_URL)
    await async_index.create(overwrite=True)

    batches = [products[i:i+batch_size] for i in range(0, len(products), batch_size)]
    t0 = time.perf_counter()
    await asyncio.gather(*[load_batch_async(async_index, b) for b in batches])
    elapsed = time.perf_counter() - t0

    await async_index.client.aclose()
    return len(products) / elapsed

print(f"{'Batch Size':>12} | {'Throughput (docs/sec)':>22}")
print("-" * 38)
for bs in [10, 25, 50, 100, 250]:
    tput = await benchmark_batch_size(bs)
    print(f"{bs:>12} | {tput:>22.0f}")

print("\n💡 Observation: there is usually an optimal batch size — find it!")

  Batch Size |  Throughput (docs/sec)
--------------------------------------
          10 |                   4554
          25 |                  11320
          50 |                  13693
         100 |                   6449
         250 |                  20821

💡 Observation: there is usually an optimal batch size — find it!


## ✅ Checkpoint

In [11]:
# Cell 10: Checkpoint
score = 0

# Test 1: async was faster than sync
if async_time < sync_time:
    print("✅ Test 1 PASS: async ingestion is faster than sync")
    score += 1
else:
    print("❌ Test 1 FAIL: async should be faster — check REDIS_URL connection")

# Test 2: speedup is meaningful
if speedup >= 1.2:
    print(f"✅ Test 2 PASS: speedup is {speedup:.1f}×")
    score += 1
else:
    print(f"❌ Test 2 FAIL: speedup {speedup:.1f}× is too low — check batch size")

# Test 3: search returned results
if len(results) == 3:
    print("✅ Test 3 PASS: async query returned 3 results")
    score += 1
else:
    print(f"❌ Test 3 FAIL: expected 3 results, got {len(results)}")

print(f"\n🏆 Score: {score}/3")

❌ Test 1 FAIL: async should be faster — check REDIS_URL connection
❌ Test 2 FAIL: speedup 0.7× is too low — check batch size
✅ Test 3 PASS: async query returned 3 results

🏆 Score: 1/3


---
## 📝 Assignment: Retry Logic for Production Ingestion

**Scenario**: You are ingesting 5,000 product records from a live data feed.
The Redis connection occasionally drops for a brief moment (network blip).
Your pipeline must handle this gracefully — not crash.

**Task 1**: Implement `load_with_retry()` with exponential backoff.

**Task 2**: Implement `stress_test()` — run 50 parallel batches and print a report.

**Task 3 (Bonus)**: Simulate a failure — manually raise an exception 20% of the time,
confirm that retry logic handles it and all documents are eventually indexed.

In [ ]:
# ── ASSIGNMENT TASK 1 ──────────────────────────────────────────────────────
import random

async def load_with_retry(index: AsyncSearchIndex, batch: list, max_retries: int = 3) -> int:
    """
    Load a batch with exponential backoff retry.

    Requirements:
    - Attempt to load the batch
    - On any exception: wait 2^attempt seconds, then retry
    - Print: "⚠️ Attempt {n} failed: {error}" for each failure
    - If all retries are exhausted: raise the last exception
    - On success: return the number of documents loaded
    """
    # TODO: implement retry logic here
    pass


# ── ASSIGNMENT TASK 2 ──────────────────────────────────────────────────────
async def stress_test(n_docs: int = 5000, batch_size: int = 100):
    """
    Run a full stress test with retry-enabled ingestion.

    Requirements:
    - Generate n_docs random products
    - Split into batches of batch_size
    - Run all batches in parallel using load_with_retry
    - Print a summary:
        Total docs   : 5000
        Total time   : X.XXs
        Throughput   : XXXX docs/sec
        Total retries: X
    """
    # TODO: implement stress test here
    pass


# ── ASSIGNMENT TASK 3 (Bonus) ─────────────────────────────────────────────
async def load_with_simulated_failure(index, batch, failure_rate=0.2, max_retries=3):
    """
    BONUS: Simulate random failures.

    Before calling index.load(), randomly raise an Exception
    with probability failure_rate. Use random.random() < failure_rate.
    Confirm that retry logic in load_with_retry handles it correctly.
    """
    # TODO: implement simulated failure + retry
    pass


# Run your stress test:
# await stress_test(n_docs=500, batch_size=50)

print("📝 Assignment cells ready — implement the TODO functions above")